# Analysis of Barren Plateau Mitigation Techniques with Qubit Scaling

This implementation analyzes barren plateau behavior across different system sizes
by running VQE experiments with varying numbers of qubits.

Key Features:
- Runs experiments for multiple qubit numbers
- Analyzes variance scaling with system size
- Compares all VQE methods across different scales
- Generates comprehensive scaling plots


In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.append("./../")

from src.barren_plateau_analyzer import BarrenPlateauAnalyzer
from qubap.qiskit.state_efficient_ansatz import ansatz_constructor
from qiskit.circuit.library import EfficientSU2

In [ ]:
# Configuration
QUBIT_RANGE = [2, 3, 4, 5, 6]
NUM_ITERATIONS = 15
PLOTS_DIR = "./scaling_analysis"

print("=" * 70)
print("VQE BARREN PLATEAU SCALING ANALYSIS")
print("=" * 70)
print(f"Qubit range: {QUBIT_RANGE}")
print(f"Iterations: {NUM_ITERATIONS}")
print(f"Output: {PLOTS_DIR}")

Path(PLOTS_DIR).mkdir(parents=True, exist_ok=True)

def debug_sea_ansatz(num_qubits):
    """Debug SEA ansatz construction for specific qubit number."""
    print(f"   Debugging SEA ansatz for {num_qubits} qubits...")
    
    # Try different deep configurations
    configs = [
        [1, 1, 1],
        [1, 1], 
        [1],
        [2] if num_qubits >= 2 else [1],
        [1] * min(num_qubits, 3)  # Adaptive config
    ]
    
    for i, deep_config in enumerate(configs):
        try:
            print(f"    Trying config {i+1}: deep={deep_config}")
            ansatz = ansatz_constructor(num_qubits, deep=deep_config, set_barrier=True)
            print(f"     Success: {ansatz.num_parameters} parameters")
            return ansatz
        except Exception as e:
            print(f"     Failed: {e}")
            continue
    
    # Fallback to EfficientSU2
    print(f"    Using EfficientSU2 fallback...")
    return EfficientSU2(num_qubits, ["ry", "rz"], "linear", 1)

def create_safe_analyzer(num_qubits):
    """Create analyzer with safe ansatz construction."""
    
    class SafeAnalyzer(BarrenPlateauAnalyzer):
        def setup_ansatz(self):
            print(" Setting up ansatz with safety checks...")
            
            # Standard ansatz - usually works
            self.ansatz_standard = EfficientSU2(
                self.num_qubits, ["ry", "rz"], "circular", 0
            ).decompose()
            print(f"   Standard: {self.ansatz_standard.num_parameters} params")
            
            # Debug and create safe SEA ansatz
            self.ansatz_sea = debug_sea_ansatz(self.num_qubits)
            print(f"   SEA: {self.ansatz_sea.num_parameters} params")
            
            # MPS ansatzes with error handling
            try:
                from qubap.qiskit.mps_pretraining import Ansatz
                self.ansatz_mps = Ansatz(self.num_qubits, diagonal=True)
                self.ansatz_full = Ansatz(self.num_qubits, diagonal=False)
                print(f"   MPS: {self.ansatz_mps.num_parameters}, {self.ansatz_full.num_parameters} params")
            except Exception as e:
                print(f"    MPS failed, using EfficientSU2 fallback: {e}")
                self.ansatz_mps = EfficientSU2(self.num_qubits, ["ry"], "linear", 1)
                self.ansatz_full = EfficientSU2(self.num_qubits, ["ry", "rz"], "linear", 2)
    
    return SafeAnalyzer(num_qubits=num_qubits)

def run_analysis(num_qubits, num_iterations):
    """Run analysis for specific qubit number."""
    print(f"\n{'='*40}")
    print(f" Analyzing {num_qubits} qubits")
    print(f"{'='*40}")
    
    try:
        analyzer = create_safe_analyzer(num_qubits)
        results = analyzer.run_complete_analysis(num_iters=num_iterations)
        
        if not results:
            print(f" No results for {num_qubits} qubits")
            return None
        
        # Extract metrics
        summary = {}
        for method_name, data in results.items():
            try:
                summary[method_name] = {
                    'num_qubits': num_qubits,
                    'gradient_variance': data['bp_diagnostics']['gradient_variance'],
                    'gradient_norm_mean': data['bp_diagnostics']['gradient_norm_mean'],
                    'final_energy_error': data['performance_metrics']['final_energy_error'],
                    'state_fidelity': data['performance_metrics']['state_fidelity'],
                    'num_parameters': len(data['method_results']['final_params'])
                }
            except Exception as e:
                print(f"    Error extracting {method_name}: {e}")
                continue
        
        print(f" Success: {len(summary)} methods for {num_qubits} qubits")
        return summary
        
    except Exception as e:
        print(f" Failed {num_qubits} qubits: {e}")
        return None

# Run analysis for all qubit numbers
print(f"\n Starting analysis...")
all_results = {}

for num_qubits in QUBIT_RANGE:
    results = run_analysis(num_qubits, NUM_ITERATIONS)
    all_results[num_qubits] = results
    
    if results:
        # Save intermediate results
        result_file = os.path.join(PLOTS_DIR, f'results_{num_qubits}qubits.json')
        with open(result_file, 'w') as f:
            json.dump(results, f, indent=2, default=str)
        print(f" Saved: {result_file}")

# Convert to DataFrame
print(f"\n Processing results...")
data_rows = []
for num_qubits, methods_data in all_results.items():
    if methods_data is None:
        continue
    for method_name, metrics in methods_data.items():
        row = {'num_qubits': num_qubits, 'method': method_name, **metrics}
        data_rows.append(row)

df = pd.DataFrame(data_rows)

if df.empty:
    print(" No data to analyze")
    exit()

print(f" Dataset: {len(df)} data points")
print(f"   Methods: {list(df['method'].unique())}")
print(f"   Systems: {sorted(df['num_qubits'].unique())}")

# Save complete data
df.to_csv(os.path.join(PLOTS_DIR, 'complete_data.csv'), index=False)
print(f" Saved: complete_data.csv")

# Plot variance scaling
print(f"\n Creating variance scaling plot...")
plt.figure(figsize=(10, 8))

methods = df['method'].unique()
colors = plt.cm.Set1(np.linspace(0, 1, len(methods)))
base_markers = ['o', 's', '^', 'D', 'v', '*', 'p', 'h', '+', 'x']
markers = [base_markers[i % len(base_markers)] for i in range(len(methods))]

for i, (method, color) in enumerate(zip(methods, colors)):
    method_data = df[df['method'] == method].sort_values('num_qubits')
    if len(method_data) > 0:
        plt.semilogy(method_data['num_qubits'], 
                    method_data['gradient_variance'], 
                    marker=markers[i % len(markers)],
                    color=color,
                    label=method, 
                    markersize=8, 
                    linewidth=2,
                    alpha=0.8)

# # Add theoretical lines
# qubits_range = np.array(sorted(df['num_qubits'].unique()))
# if len(qubits_range) > 1:
#     for alpha in [0.3, 0.5, 0.7]:
#         theoretical = 1e-2 * np.exp(-alpha * qubits_range)
#         plt.semilogy(qubits_range, theoretical, '--', 
#                     alpha=0.5, color='gray',
#                     label=f'Theory: exp(-{alpha}n)')

plt.xlabel('Number of Qubits', fontsize=14)
plt.ylabel('Gradient Variance', fontsize=14)
plt.title('Gradient Variance Scaling with System Size', fontsize=16, fontweight='bold')
plt.xticks(sorted(df['num_qubits'].unique()))  # Force integer x-axis values
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()

variance_plot = os.path.join(PLOTS_DIR, 'variance_scaling.pdf')
plt.savefig(variance_plot, dpi=300, bbox_inches='tight')
print(f" Saved: {variance_plot}")
plt.show()

# Create summary table
print(f"\n Creating summary table...")
try:
    variance_pivot = df.pivot(index='num_qubits', columns='method', values='gradient_variance')
    
    csv_file = os.path.join(PLOTS_DIR, 'summary_table.csv')
    variance_pivot.to_csv(csv_file)
    print(f" Saved: {csv_file}")
    
    print(f"\n VARIANCE SCALING SUMMARY")
    print("=" * 60)
    print(variance_pivot.to_string(float_format='%.2e'))
    print("=" * 60)
    
except Exception as e:
    print(f" Summary table error: {e}")

# Final analysis
print(f"\n Results:")
successful_systems = len([r for r in all_results.values() if r is not None])
print(f"  Successful systems: {successful_systems}/{len(QUBIT_RANGE)}")

if not df.empty:
    # Best method at largest system
    largest_system = df['num_qubits'].max()
    largest_data = df[df['num_qubits'] == largest_system]
    if len(largest_data) > 0:
        best_method = largest_data.loc[largest_data['gradient_variance'].idxmin(), 'method']
        best_variance = largest_data['gradient_variance'].min()
        print(f"  Best method at {largest_system} qubits: {best_method}")
        print(f"  Lowest gradient variance: {best_variance:.2e}")

VQE BARREN PLATEAU SCALING ANALYSIS
Qubit range: [2, 3, 4, 5, 6]
Iterations: 15
Output: ./scaling_analysis

 Starting analysis...

 Analyzing 2 qubits
Setting up Hamiltonian...
0.75 * II
- 0.25 * IZ
- 0.25 * ZI
- 0.25 * ZZ
Hamiltonian terms: 4
Exact ground state energy: 0j
 Setting up ansatz with safety checks...
   Standard: 4 params
   Debugging SEA ansatz for 2 qubits...
    Trying config 1: deep=[1, 1, 1]
     Success: 6 parameters
   SEA: 6 params
   MPS: 15, 15 params
COMPREHENSIVE VQE BARREN PLATEAU ANALYSIS
Running Standard VQE...

Standard VQE completed successfully!
Running Local-Global VQE...

Local-Global VQE completed successfully!
Running Adiabatic VQE...

Adiabatic VQE completed successfully!
Running VQE with SEA...

VQE with SEA completed successfully!
Running Pretrained VQE...
  Pretrained VQE results structure: ['pretrain', 'full']
  Computed 15 MPS energies
  Computed 15 full VQE energies

Pretrained VQE completed successfully!
 Success: 5 methods for 2 qubits
 Saved